In [ ]:
##네이버 영화리뷰 감정 분석하기 by SentencePeice

In [ ]:
# sentencepiece 설치하기

In [1]:
pip install sentencepiece --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 데이터 준비하기

In [2]:
import pandas as pd

train_data = pd.read_csv('ratings_train.txt', sep='\t')
train_data = train_data.dropna(how='any')

# SentencePiece 학습용 텍스트 파일 만들기
with open('nsmc_corpus.txt', 'w', encoding='utf-8') as f:
    for sentence in train_data['document']:
        f.write(sentence + '\n')

In [ ]:
# 모델 학습시키기

In [3]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='nsmc_corpus.txt',
    model_prefix='nsmc_sp',      # 결과 파일 이름
    vocab_size=8000,              # 단어 사전 크기 (조절 가능한 하이퍼파라미터)
    model_type='bpe',             # 'bpe', 'unigram', 'char', 'word' 중 선택 가능
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)
# nsmc_sp.model, nsmc_sp.vocab 파일 생성

I0000 00:00:1788333585.015681 3651853 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: nsmc_corpus.txt
  input_format: 
  model_prefix: nsmc_sp
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_diffe

True

In [ ]:
# 학습된 모델 불러오기

In [4]:
sp = spm.SentencePieceProcessor()
sp.load('nsmc_sp.model')

True

In [5]:
print(sp.encode_as_pieces("자연어처리가 너무 재밌어서 밥 먹는 것도 가끔 까먹어요")) #테스트

['▁자연', '어', '처', '리가', '▁너무', '▁재밌어서', '▁밥', '▁먹는', '▁것도', '▁가끔', '▁까', '먹', '어요']


In [ ]:
# sp_tokenize -> sentencepiece 버전으로 바꾸기

In [6]:
def sp_tokenize(sp, corpus):
    tensor = []
    for sentence in corpus:
        # 문장을 SentencePiece로 토큰화 + 정수 인덱스로 변환
        tokens = sp.encode_as_ids(sentence)
        tensor.append(torch.tensor(tokens, dtype=torch.long))
    
    # pad_sequence로 길이 맞춰주기
    tensor = pad_sequence(tensor, batch_first=True, padding_value=0)
    return tensor, sp

In [ ]:
# pad_sequence 
문장 1 → [12, 35, 8]
문장 2 → [7, 91, 23, 45, 6]
문장 3 → [18, 4]

같은 batch에 한 번에 처리하기 위해 길이 맞춰줌
[12, 35,  8,  0,  0]
[ 7, 91, 23, 45,  6]
[18,  4,  0,  0,  0]

In [ ]:
# 새 토크나이저로 데이터 만들기

In [91]:
corpus = train_data['document'].tolist()
tensor, sp_tokenizer = sp_tokenize(sp, corpus)

print(sp.get_piece_size()) # 단어 사전 크기
print(tensor.shape) # Tensor shape

8000
torch.Size([149995, 130])


In [ ]:
# pytorch 불러오기

In [8]:
import torch
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import sentencepiece as spm

In [ ]:
# sp_tokenize 다시 재정의

In [9]:
def sp_tokenize(sp, corpus):
    tensor = [] #토큰화한 결과 담을 빈 list
    for sentence in corpus:
        tokens = sp.encode_as_ids(sentence)
        tensor.append(torch.tensor(tokens, dtype=torch.long))
    
    tensor = pad_sequence(tensor, batch_first=True, padding_value=0)
    return tensor, sp

In [ ]:
# sentencepiece model 다시 불러오기 

In [10]:
sp = spm.SentencePieceProcessor()
sp.load('nsmc_sp.model')

True

In [ ]:
# 새 토크나이저로 데이터 만들기

In [90]:
corpus = train_data['document'].tolist()
tensor, sp_tokenizer = sp_tokenize(sp, corpus)

print(sp.get_piece_size()) # SentencePiece 단어 사전 크기
print(tensor.shape) # Tensor shape

8000
torch.Size([149995, 130])


In [ ]:
# SentencePiece 학습용 임시 텍스트 파일 만들기

In [15]:
temp_file = 'korean-english-park.train.ko.temp'

In [ ]:
# 문장들 불러와 corpus에 저장

In [21]:
with open('korean-english-park.train.ko', 'r', encoding='utf-8') as f:
    corpus = f.readlines()

print(len(corpus))
print(corpus[:5])

94123
['개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"\n', '모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하지 않는다.\n', '그러나 이것은 또한 책상도 필요로 하지 않는다.\n', '79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분이든 그 움직임에따라 커서의 움직임을 조절하는 회전 운동 센서를 사용하고 있다.\n', '정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔음을 밝혔으며, 세계 해상 교역량의 거의 3분의 1을 운송하는 좁은 해로인 말라카 해협이 테러 공격을 당하기 쉽다고 경고하고 있다.\n']


In [ ]:
# \n, 빈문장 제외

In [23]:
import re

filtered_corpus = []

for sentence in corpus:
    sentence = sentence.strip()  # 줄바꿈(\n) 제거
    
    if len(sentence) == 0:  # 빈 문장 제외
        continue
    
    filtered_corpus.append(sentence)

print(len(filtered_corpus))
print(filtered_corpus[:5])

94123
['개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"', '모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하지 않는다.', '그러나 이것은 또한 책상도 필요로 하지 않는다.', '79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분이든 그 움직임에따라 커서의 움직임을 조절하는 회전 운동 센서를 사용하고 있다.', '정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔음을 밝혔으며, 세계 해상 교역량의 거의 3분의 1을 운송하는 좁은 해로인 말라카 해협이 테러 공격을 당하기 쉽다고 경고하고 있다.']


In [ ]:
# 학습용 텍스트 파일 만들기

In [24]:
temp_file = 'korean-english-park.train.ko.temp'
vocab_size = 8000

with open(temp_file, 'w') as f:
    for row in filtered_corpus:
        f.write(str(row) + '\n')

In [89]:
# 원본 다시 불러오기
with open('korean-english-park.train.ko', 'r', encoding='utf-8') as f:
    corpus = f.readlines()

# 정제 (기본적인 것만)
filtered_corpus = [line.strip() for line in corpus if len(line.strip()) > 0]

print(len(filtered_corpus))
print(filtered_corpus[:5])

# temp 파일 다시 쓰기
temp_file = 'korean-english-park.train.ko.temp'
with open(temp_file, 'w') as f:
    for row in filtered_corpus:
        f.write(str(row) + '\n')

94123
['개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"', '모든 광마우스와 마찬가지 로 이 광마우스도 책상 위에 놓는 마우스 패드를 필요로 하지 않는다.', '그러나 이것은 또한 책상도 필요로 하지 않는다.', '79.95달러하는 이 최첨단 무선 광마우스는 허공에서 팔목, 팔, 그외에 어떤 부분이든 그 움직임에따라 커서의 움직임을 조절하는 회전 운동 센서를 사용하고 있다.', '정보 관리들은 동남 아시아에서의 선박들에 대한 많은 (테러) 계획들이 실패로 돌아갔음을 밝혔으며, 세계 해상 교역량의 거의 3분의 1을 운송하는 좁은 해로인 말라카 해협이 테러 공격을 당하기 쉽다고 경고하고 있다.']


In [ ]:
# SentencePiece 새로운 토크나이저 학습

In [26]:
import sentencepiece as spm

vocab_size = 8000

spm.SentencePieceTrainer.Train(
    '--input={} --model_prefix=korean_spm --vocab_size={}'.format(temp_file, vocab_size)
)

I0000 00:00:1788334942.891020 3651853 sentencepiece_trainer.cc:227] Running command: --input=korean-english-park.train.ko.temp --model_prefix=korean_spm --vocab_size=8000
I0000 00:00:1788334942.896978 3651853 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: korean-english-park.train.ko.temp
  input_format: 
  model_prefix: korean_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 

True

In [ ]:
# 파일 확인하기 

In [28]:
!ls -l korean_spm*

-rw-r--r--  1 sooah  staff  380795  9월  2 16:49 korean_spm.model
-rw-r--r--  1 sooah  staff  147822  9월  2 16:49 korean_spm.vocab


In [29]:
sp = spm.SentencePieceProcessor()
sp.load('korean_spm.model')

True

In [ ]:
# 토크나이저 확인하기

In [30]:
p = spm.SentencePieceProcessor()
sp.load('korean_spm.model')

print(sp.encode_as_pieces(filtered_corpus[0]))

['▁개인', '용', '▁컴퓨터', '▁사용', '의', '▁상', '당', '▁부분', '은', '▁"', '이', '것', '보다', '▁뛰어', '날', '▁수', '▁있', '느냐', '?', '"']


In [ ]:
# torch 불러옴, 문장들 tensor로 변환

In [31]:
import torch
from torch.nn.utils.rnn import pad_sequence

def sp_tokenize(sp, corpus):
    tensor = []
    for sentence in corpus:
        tokens = sp.encode_as_ids(str(sentence))
        tensor.append(torch.tensor(tokens, dtype=torch.long))
    tensor = pad_sequence(tensor, batch_first=True, padding_value=0)
    return tensor, sp

In [ ]:
# 단어 사전 크기 확인

In [88]:
tensor, sp_tokenizer = sp_tokenize(sp, filtered_corpus)
print(sp.get_piece_size()) # 단어 사전 크기
print(tensor.shape) # Tensor shape

8000
torch.Size([94123, 233])


In [ ]:
# 단계별 확인

In [87]:
print(filtered_corpus[0]) # 원본문장
print(sp.encode_as_pieces(filtered_corpus[0])) # 토큰화
print(tensor[0]) # 텐서

개인용 컴퓨터 사용의 상당 부분은 "이것보다 뛰어날 수 있느냐?"
['▁개인', '용', '▁컴퓨터', '▁사용', '의', '▁상', '당', '▁부분', '은', '▁"', '이', '것', '보다', '▁뛰어', '날', '▁수', '▁있', '느', '냐', '?', '"']
tensor([   8, 1078,    4,   55, 2393,   41, 2104,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    

In [ ]:
# NSMC train/test 데이터 준비

In [36]:
import pandas as pd

train_data = pd.read_csv('ratings_train.txt', sep='\t').dropna()
test_data = pd.read_csv('ratings_test.txt', sep='\t').dropna()

print(len(train_data), len(test_data))

149995 49997


In [ ]:
# sp_tokenize()로 train/test 각각 토큰화

In [37]:
import torch
from torch.nn.utils.rnn import pad_sequence

def sp_tokenize(sp, corpus, max_len=None):
    tensor = []
    for sentence in corpus:
        tokens = sp.encode_as_ids(str(sentence))
        tensor.append(torch.tensor(tokens, dtype=torch.long))
    tensor = pad_sequence(tensor, batch_first=True, padding_value=0)
    return tensor, sp #pad로 인해 긴 토큰 130개에 맞춰 15만개 문장에 0이 채워짐

train_tensor, sp = sp_tokenize(sp, train_data['document'].tolist())
test_tensor, sp = sp_tokenize(sp, test_data['document'].tolist())

train_labels = torch.tensor(train_data['label'].tolist(), dtype=torch.long)
test_labels = torch.tensor(test_data['label'].tolist(), dtype=torch.long)

In [ ]:
# Text Classifier 모델 만들기

In [38]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, tensor, labels):
        self.tensor = tensor
        self.labels = labels
    def __len__(self):
        return len(self.tensor)
    def __getitem__(self, idx):
        return self.tensor[idx], self.labels[idx]

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128): #130개 토큰, 128차원 벡터
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True) #LSTM 사용
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x) 
        _, (hidden, _) = self.lstm(x) #문장 130개 순차적으로 처리
        hidden = hidden.squeeze(0)
        return self.fc(hidden)

In [ ]:
#학습 및 평가

In [40]:
train_dataset = NSMCDataset(train_tensor, train_labels)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = NSMCDataset(test_tensor, test_labels)
test_loader = DataLoader(test_dataset, batch_size=64)

model = TextClassifier(vocab_size=sp.get_piece_size())
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [41]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            pred = model(x)
            predicted = torch.argmax(pred, dim=1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    model.train()
    return correct / total

for epoch in range(5):
    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
    
    test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}, Test Accuracy: {test_acc:.4f}")

Epoch 1, Test Accuracy: 0.4965
Epoch 2, Test Accuracy: 0.5035
Epoch 3, Test Accuracy: 0.4965
Epoch 4, Test Accuracy: 0.5035
Epoch 5, Test Accuracy: 0.5035


In [ ]:
# ->accuracy 저조함

In [ ]:
# train label, tensor 길이 확인

In [43]:
print(len(train_tensor))
print(len(train_labels))

149995
149995


In [ ]:
# 순서 맞는지 확인

In [49]:
print(train_data['document'].iloc[0])   # 원본 첫 문장
print(train_data['label'].iloc[0])       # 그 문장의 라벨

아 더빙.. 진짜 짜증나네요 목소리
0


In [ ]:
# 데이터 짝 확인

In [86]:
print(train_data['document'].iloc[0])
print(train_data['label'].iloc[0])
print(train_tensor[0][:20])  # 처음 20개 토큰만
print(train_labels[0])

아 더빙.. 진짜 짜증나네요 목소리
0
tensor([   8, 1078,    4,   55, 2393,   41, 2104,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0])
tensor(0)


In [ ]:
# vocab size 확인

In [85]:
print(sp.get_piece_size())
print(vocab_size)

8000
8000


In [ ]:
# loss값 확인하기

In [52]:
for epoch in range(5):
    total_loss = 0
    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Epoch 1, Loss: 0.6933, Test Accuracy: 0.4965
Epoch 2, Loss: 0.6932, Test Accuracy: 0.4965
Epoch 3, Loss: 0.6932, Test Accuracy: 0.4965
Epoch 4, Loss: 0.6932, Test Accuracy: 0.4965
Epoch 5, Loss: 0.6932, Test Accuracy: 0.5035


In [ ]:
# loss값이 높은 것 확인, accuracy는 여전히 낮음

In [ ]:
# 각 가중치에 대한 기울기 계산 후 업데이트

In [56]:
x_sample, y_sample = next(iter(train_loader))

optimizer.zero_grad()
pred = model(x_sample)
loss = loss_fn(pred, y_sample)
loss.backward()

In [57]:
# 각 파라미터의 기울기 확인
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"{name}: grad mean = {param.grad.abs().mean().item():.6f}")
    else:
        print(f"{name}: 기울기가 None입니다!")

embedding.weight: grad mean = 0.000000
lstm.weight_ih_l0: grad mean = 0.000000
lstm.weight_hh_l0: grad mean = 0.000000
lstm.bias_ih_l0: grad mean = 0.000002
lstm.bias_hh_l0: grad mean = 0.000002
fc.weight: grad mean = 0.000005
fc.bias: grad mean = 0.119583


In [ ]:
# 출력층에 가까운 fc.bias만 기울기가 살아 있고 나머지는 0에 가까움->기울기 소실

In [ ]:
# LSTM 초기 가중치가 설정 확인을 위해 구조 단순화하기, 평균 풀링 방식으로 바꾸기
기존 모델
토큰 → Embedding → LSTM → 마지막 hidden → FC(Fully connected layer) → 분류

지금 모델
토큰 → Embedding → 평균(mean) → FC → 분류

In [58]:
class SimpleTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.fc1 = nn.Linear(embedding_dim, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)         # LSTM 대신, 그냥 단어 벡터들의 평균을 냄
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleTextClassifier(vocab_size=sp.get_piece_size()) #SimpleTextClassifier
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Test Accuracy: {evaluate(model, test_loader):.4f}")

Epoch 1, Loss: 0.4398, Test Accuracy: 0.8409
Epoch 2, Loss: 0.3326, Test Accuracy: 0.8477
Epoch 3, Loss: 0.3150, Test Accuracy: 0.8500
Epoch 4, Loss: 0.3060, Test Accuracy: 0.8495
Epoch 5, Loss: 0.2987, Test Accuracy: 0.8497


In [ ]:
# ->Epoch 3에서 최고점 0.85를 찍고 이후로 정체되거나 떨어지는 신호. 3번째에서 멈추는 게 최적일 수도 있겠다.

In [ ]:
#다시 기울기 확인해보기

In [59]:
x_sample, y_sample = next(iter(train_loader))

optimizer.zero_grad()
pred = model(x_sample)
loss = loss_fn(pred, y_sample)
loss.backward()

for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"{name}: grad mean = {param.grad.abs().mean().item():.6f}")

embedding.weight: grad mean = 0.000007
fc1.weight: grad mean = 0.000508
fc1.bias: grad mean = 0.007984
fc2.weight: grad mean = 0.000850
fc2.bias: grad mean = 0.015961


In [ ]:
# Mecab으로 데이터 토큰화하기

In [60]:
from konlpy.tag import Mecab

mecab = Mecab('/opt/homebrew/lib/mecab/dic/mecab-ko-dic')

# train, test 각각 형태소 분석
train_split = [mecab.morphs(str(s)) for s in train_data['document']]
test_split = [mecab.morphs(str(s)) for s in test_data['document']]

print(train_split[0])

['아', '더', '빙', '.', '.', '진짜', '짜증', '나', '네요', '목소리']


In [ ]:
# 토크나이저 

In [67]:
import torch
from torch.nn.utils.rnn import pad_sequence

class Tokenizer:
    def __init__(self, filters=''): # 사전 준비 
        self.word_index = {}
        self.index_word = {}
        self.filters = filters

    def fit_on_texts(self, corpus): # 코퍼스 보고 사전 만들기
        for sentence in corpus:
            tokens = sentence.split() if isinstance(sentence, str) else sentence
            for token in tokens:
                if token not in self.word_index:
                    self.word_index[token] = len(self.word_index) + 1
        self.index_word = {idx: word for word, idx in self.word_index.items()}

    def texts_to_sequences(self, corpus): # 문장을 숫자로
        sequences = []
        for sentence in corpus:
            tokens = sentence.split() if isinstance(sentence, str) else sentence
            seq = [self.word_index.get(token, 0) for token in tokens]
            sequences.append(torch.tensor(seq, dtype=torch.long))
        return sequences

    def sequences_to_texts(self, sequences): #숫자를 함수로
        texts = []
        for seq in sequences:
            if isinstance(seq, torch.Tensor):
                seq = seq.tolist()
            tokens = [self.index_word.get(idx, "") for idx in seq if idx != 0]
            texts.append(tokens)
        return texts

In [ ]:
# train_split으로 사전 만들기 

In [84]:
mecab_tokenizer = Tokenizer()
mecab_tokenizer.fit_on_texts(train_split)

mecab_train_seq = mecab_tokenizer.texts_to_sequences(train_split)
mecab_test_seq = mecab_tokenizer.texts_to_sequences(test_split)

mecab_train_tensor = pad_sequence(mecab_train_seq, batch_first=True, padding_value=0)
mecab_test_tensor = pad_sequence(mecab_test_seq, batch_first=True, padding_value=0)

mecab_vocab_size = len(mecab_tokenizer.word_index) + 1

print(mecab_vocab_size)
print(mecab_train_tensor.shape)

54045
torch.Size([149995, 116])


In [ ]:
# train, test 길이 확인하기

In [83]:
print(mecab_train_tensor.shape[1])
print(mecab_test_tensor.shape[1])

116
116


In [ ]:
# 두 개 길이 맞춰주기

In [75]:
if mecab_train_tensor.shape[1] != mecab_test_tensor.shape[1]:
    max_len = mecab_train_tensor.shape[1]
    # test를 train 길이에 맞춰 자르거나 채우기
    mecab_test_tensor = torch.zeros(len(mecab_test_seq), max_len, dtype=torch.long)
    for i, seq in enumerate(mecab_test_seq):
        length = min(len(seq), max_len)
        mecab_test_tensor[i, :length] = seq[:length]

print(mecab_test_tensor.shape)

torch.Size([49997, 116])


In [ ]:
# SimpleTextClassifier 재사용(같은 모델 구조로 학습)

In [71]:
mecab_train_dataset = NSMCDataset(mecab_train_tensor, train_labels)
mecab_train_loader = DataLoader(mecab_train_dataset, batch_size=64, shuffle=True)

mecab_test_dataset = NSMCDataset(mecab_test_tensor, test_labels)
mecab_test_loader = DataLoader(mecab_test_dataset, batch_size=64)

mecab_model = SimpleTextClassifier(vocab_size=mecab_vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mecab_model.parameters(), lr=0.001)

mecab_accuracy_history = []

for epoch in range(5):
    total_loss = 0
    for x, y in mecab_train_loader:
        optimizer.zero_grad()
        pred = mecab_model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    test_acc = evaluate(mecab_model, mecab_test_loader)
    mecab_accuracy_history.append(test_acc)
    print(f"[Mecab] Epoch {epoch+1}, Loss: {total_loss/len(mecab_train_loader):.4f}, Test Accuracy: {test_acc:.4f}")

[Mecab] Epoch 1, Loss: 0.4253, Test Accuracy: 0.8404
[Mecab] Epoch 2, Loss: 0.3270, Test Accuracy: 0.8478
[Mecab] Epoch 3, Loss: 0.2944, Test Accuracy: 0.8483
[Mecab] Epoch 4, Loss: 0.2711, Test Accuracy: 0.8474
[Mecab] Epoch 5, Loss: 0.2526, Test Accuracy: 0.8475


In [76]:
print(f"SentencePiece 단어 사전 크기: {sp.get_piece_size()}")
print(f"Mecab 단어 사전 크기: {mecab_vocab_size}")
print()
print(f"SentencePiece 최고 Test Accuracy: {max([0.8409, 0.8477, 0.8500, 0.8495, 0.8497]):.4f}")
print(f"Mecab 최고 Test Accuracy: {max(mecab_accuracy_history):.4f}")

SentencePiece 단어 사전 크기: 8000
Mecab 단어 사전 크기: 54045

SentencePiece 최고 Test Accuracy: 0.8500
Mecab 최고 Test Accuracy: 0.8483
